# Initial-shock anomaly errors: RMSE and MAE

Archive-backed companion to `6a_initial_shock_std_index.ipynb`.

**Configure → inventory and cache plan → reuse/prepare common-grid global indices →
full-cohort anomaly calculation → cached RMSE/MAE → case comparisons.**

This applies the metrics in `Compute_RMSE_With_MAE_Index_share.ncl` to the E3SM
and CESM-SMYLE archives: ensemble mean, consecutive monthly blocks, area-weighted
global indices, independent model/observation climatologies, then RMSE and MAE.


In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

repo_root = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / 'esp_lab').is_dir()), None)
if repo_root is not None:
    sys.path.insert(0, str(repo_root))
from workflows.diagnostics.initial_shock_error_archive import plan_archive_run, compute_archive_plan
from esp_lab.diagnostics.initial_shock_error import (
    plot_rmse_mae, NCL_RMSE_RANGES, NCL_MAE_RANGES,
)
from esp_lab.utils.dask_util import DaskConfig, get_cluster_client
from esp_lab.utils.resource_utils import ResourceTracker


## User setup and workflow configuration

Select a field and E3SM cases as in `6a`. Input discovery, unit conversion,
verification-time checks, conservative regridding, and ensemble completeness checks
are shared with that workflow. `TREFHT` matches the NCL example.

The NCL script uses five annual samples. The current archive provides 24 months, so
the defaults apply the same RMSE/MAE definitions to two consecutive 12-month blocks.
May blocks are May–April and November blocks are November–October. The plot uses the
NCL temperature thresholds; adjust `RMSE_RANGES` and `MAE_RANGES` for other fields.


In [ ]:
VAR_CONFIG = {
    'TREFHT': dict(obs_product='ERA5', obs_variable='tas', units='degC',
                   model_scale=1., model_offset=-273.15, smyle_scale=1., smyle_offset=-273.15,
                   obs_scale=1., obs_offset=-273.15),
    'TS': dict(obs_product='ERA5', obs_variable='ts', units='degC',
               model_scale=1., model_offset=-273.15, smyle_scale=1., smyle_offset=-273.15,
               obs_scale=1., obs_offset=-273.15),
    'PRECT': dict(obs_product='GPCP_v2.3', obs_variable='PRECT', units='mm/day',
                  model_scale=86400000., model_offset=0., smyle_scale=86400000., smyle_offset=0.,
                  obs_scale=1., obs_offset=0.),
    'PSL': dict(obs_product='ERA5', obs_variable='psl', units='hPa',
                model_scale=.01, model_offset=0., smyle_scale=.01, smyle_offset=0.,
                obs_scale=.01, obs_offset=0.),
}
field = 'TREFHT'
variable = dict(VAR_CONFIG[field], field=field, model_variable=field, smyle_variable=field)

E3SM_CASES = {
    'E3SM-FOSIRL': dict(case_prefix='WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL', cache_tag='JRA55_FOSIRL'),
    'E3SM-Reanalysis': dict(case_prefix='WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce', cache_tag='Reanalysis'),
    'E3SM-4DEnVarOcn': dict(case_prefix='WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_4DEnVarOcn', cache_tag='4DEnVarOcn'),
}
WORKFLOW_SETTINGS = {
    'paths': {
        's2d_diag_root': '/global/cfs/cdirs/e3sm/S2S2D/s2d_diag',
        'figure_outdir': '/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag/initial_shock_rmse_mae',
    },
    'run': {'years': [1980, 2011], 'init_months': [5, 11], 'nlead': 24, 'smoke_mode': False},
    'e3sm': {
        'data_dir': '/global/cfs/cdirs/e3sm/S2S2D/post_process',
        'nens': 10, 'grid': '180x360_aave', 'ts_split': '2yr', 'engine': 'netcdf4',
        'chunks': {'Y': 3, 'L': 24, 'M': 2, 'lat': 90, 'lon': 180},
    },
    'smyle': {
        'include': True, 'nens': 20,
        'benchmark_dir': '/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE',
        'chunks': {'Y': 3, 'L': 24, 'M': 2, 'lat': 96, 'lon': 144},
    },
    'obs': {
        'data_dir': '/global/cfs/cdirs/e3sm/e3sm_diags/obs_for_e3sm_diags/time-series',
        'chunks': {'time': 24, 'lat': 90, 'lon': 180},
    },
    'regrid': {'target_dlat': 1., 'target_dlon': 1., 'method': 'conservative', 'periodic': True},
    'metric': {'window_months': 24, 'block_months': 12, 'start_lead': 0,
               'min_samples': None, 'min_area_fraction': .9},
    'cache': {'mode': 'auto'},  # auto / rebuild / require; inventory identity always checked
    'dask': {'enabled': True, 'workers': 8},
}
RMSE_RANGES = NCL_RMSE_RANGES
MAE_RANGES = NCL_MAE_RANGES

if WORKFLOW_SETTINGS['run']['smoke_mode']:
    E3SM_CASES = {'E3SM-FOSIRL': E3SM_CASES['E3SM-FOSIRL']}
    WORKFLOW_SETTINGS['run'].update(years=[1980, 1981], init_months=[11])
    WORKFLOW_SETTINGS['regrid'].update(target_dlat=5., target_dlon=5.)
    WORKFLOW_SETTINGS['dask']['workers'] = 2
    print('Archive-backed smoke mode: one E3SM case, CESM-SMYLE, two starts, one month.')
print('Field:', field, '| Years:', WORKFLOW_SETTINGS['run']['years'])
print('Metric:', WORKFLOW_SETTINGS['metric'])


## Inventory and cache plan

Inventory all requested initializations before expensive processing. All E3SM members
and requested starts are required. The plan identifies both the shared compact block-
index cache and the derived RMSE/MAE cache. Cache identity covers source inventories,
settings, adapters, and both algorithms. `require` opens compatible caches only.


In [ ]:
archive_plan = plan_archive_run(WORKFLOW_SETTINGS, E3SM_CASES, variable)
plan_table = pd.DataFrame([
    {'case': t['case'], 'init_month': t['month'], 'initializations': len(t['years']),
     'source_files': len(t['inventory']),
     'block_indices': 'prepare' if t['rebuild'] else 'reuse',
     'rmse_mae': 'prepare' if t['error_rebuild'] else 'reuse',
     'cache': t['error_path']}
    for t in archive_plan
])
display(plan_table)


## Dask resources

Start after the inventory succeeds. Rerunning this cell closes the previous cluster.
Run the cleanup cell when finished, including after an interrupted computation.


In [ ]:
from esp_lab.utils.notebook_resources import restart_notebook_cluster, close_notebook_resources

cluster, client, workflow_resources = restart_notebook_cluster(
    globals(),
    lambda: get_cluster_client(DaskConfig(
        cluster_type='local', workers=WORKFLOW_SETTINGS['dask']['workers']))
        if WORKFLOW_SETTINGS['dask']['enabled'] else (None, None),
)
if client is not None:
    display(client)


## Common-grid indices, anomaly metrics, and cache writing

The workflow reuses the compact global block indices from `6a` when compatible. If
needed, it opens the monthly archives, verifies represented months, converts units,
regrids model and observations, averages members and monthly blocks, and computes
area-weighted global indices.

For each case and initialization month, model and observation indices are centered on
their own means over the complete requested `Y × block` cohort, following the NCL
climatology construction. RMSE and MAE are then reduced over paired blocks within each
initialization. Derived caches retain anomalies, errors, sample counts, and provenance.


In [ ]:
comparison_by_month = compute_archive_plan(archive_plan, WORKFLOW_SETTINGS, variable)
for month, comparison in comparison_by_month.items():
    print(f'Initialization month {month:02d}')
    display(comparison[['rmse', 'mae', 'paired_sample_count', 'valid_metric']])
    display(comparison[['model_climatology', 'observation_climatology']])


## Comparison tables and NCL-style panels

Lower RMSE and MAE indicate closer anomaly evolution. Missing cells fail the configured
paired-block requirement. The two panels use the NCL thresholds and colors, while rows
represent your individual initialization years and columns represent your model cases.
May and November cohorts are centered and plotted separately.


In [ ]:
FIGURE_ROOT = Path(WORKFLOW_SETTINGS['paths']['figure_outdir'])
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
for month, comparison in comparison_by_month.items():
    import hashlib
    identities = [t['error_digest'] for t in archive_plan if t['month'] == month]
    run_id = hashlib.sha256(''.join(identities).encode()).hexdigest()[:12]
    prefix = f'{field}_init{month:02d}_{run_id}'
    summary = comparison[['rmse', 'mae', 'paired_sample_count',
                          'valid_metric']].to_dataframe().reset_index()
    display(summary)
    summary.to_csv(FIGURE_ROOT / f'{prefix}_rmse_mae_summary.csv', index=False)
    fig = plot_rmse_mae(comparison, rmse_ranges=RMSE_RANGES, mae_ranges=MAE_RANGES)
    figure_path = FIGURE_ROOT / f'{prefix}_rmse_mae.png'
    fig.savefig(figure_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print('Saved comparison:', figure_path)


## Inspect one initialization

Choose a month and initialization year from the computed cohort. These are the
full-cohort anomalies used directly by RMSE and MAE; the shaded difference is the
paired error after removing the separate model and observation climatologies.


In [ ]:
INSPECT_MONTH = WORKFLOW_SETTINGS['run']['init_months'][0]
INSPECT_YEAR = WORKFLOW_SETTINGS['run']['years'][0]
selected = comparison_by_month[INSPECT_MONTH].sel(Y=INSPECT_YEAR)
fig, axes = plt.subplots(len(selected.case), 1, figsize=(9, 3 * len(selected.case)), squeeze=False)
for ax, case in zip(axes[:, 0], selected.case.values):
    data = selected.sel(case=case)
    ax.plot(data.block, data.model_anomaly, marker='o', label='model anomaly')
    ax.plot(data.block, data.observation_anomaly, marker='o', linestyle='--', label='observation anomaly')
    ax.fill_between(data.block, data.model_anomaly, data.observation_anomaly, alpha=.15, label='error')
    ax.set(title=str(case), ylabel=variable['units'])
    ax.grid(alpha=.25)
    ax.legend(fontsize=8)
axes[-1, 0].set_xlabel('Averaging block')
fig.suptitle(f'{field}: initialization {INSPECT_YEAR}-{INSPECT_MONTH:02d}')
fig.tight_layout()
plt.show()
plt.close(fig)


## Cleanup

Cached block indices and derived metrics remain available. Source datasets are closed
inside the workflow; this cell releases the optional distributed cluster.

See [methodology](../docs/initial_shock_rmse_mae_index.md) and
[the NCL reference](../temp/Compute_RMSE_With_MAE_Index_share.ncl).


In [ ]:
close_notebook_resources(globals())
print('Closed notebook Dask resources.')
